# Fetch Missing Binance Data via API

Fetch 1h klines (00:00 UTC candle per day) for symbols missing from ClickHouse,
and save in the same format as the existing `binance_v2` parquet files.

In [8]:
from datetime import UTC
from datetime import datetime
from pathlib import Path

import pandas as pd
from binance.client import Client
from binance.enums import HistoricalKlinesType

In [2]:
# --- Configuration ---
SYMBOLS = ["FTTUSDT"]  # Add more symbols here if needed
START_DATE = "2024-12-01"
END_DATE = datetime.now(UTC).strftime("%Y-%m-%d")
OUTPUT_DIR = Path("../data/binance_v2")

print(f"Symbols: {SYMBOLS}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Output: {OUTPUT_DIR.resolve()}")

Symbols: ['FTTUSDT']
Date range: 2024-12-01 to 2026-02-24
Output: /home/ra_yeye/2026_projects/nautilus_trader/my_strategies/data/binance_v2


In [3]:
def fetch_1h_daily_klines(client: Client, symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch 1h perpetual klines from Binance API, filtered to 00:00 UTC only."""
    klines = client.get_historical_klines(
        symbol=symbol,
        interval=Client.KLINE_INTERVAL_1HOUR,
        start_str=f"{start_date} 00:00:00",
        end_str=f"{end_date} 23:59:59",
        klines_type=HistoricalKlinesType.FUTURES,
    )

    if not klines:
        return pd.DataFrame()

    df = pd.DataFrame(klines, columns=[
        "timestamp", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades_count", "taker_buy_volume",
        "taker_buy_quote_volume", "ignore",
    ])

    # Convert types
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)
    for col in ["open", "high", "low", "close", "volume", "quote_volume",
                "taker_buy_volume", "taker_buy_quote_volume"]:
        df[col] = df[col].astype(float)
    df["trades_count"] = df["trades_count"].astype(int)

    # Filter to 00:00 UTC only (1 candle per day)
    df = df[df["timestamp"].dt.hour == 0].copy()

    # Add metadata columns to match existing parquet format
    df["symbol"] = symbol
    df["exchange"] = "binance"
    df["interval"] = "1h"
    df["type"] = "PERPETUAL"

    # Reorder to match existing files
    df = df[[
        "symbol", "exchange", "interval", "timestamp", "type", "close_time",
        "open", "high", "low", "close", "volume", "quote_volume",
        "taker_buy_volume", "taker_buy_quote_volume", "trades_count",
    ]].reset_index(drop=True)

    return df

In [4]:
# Fetch data
client = Client()

for symbol in SYMBOLS:
    print(f"Fetching {symbol}...")
    df = fetch_1h_daily_klines(client, symbol, START_DATE, END_DATE)

    if df.empty:
        print(f"  No data found for {symbol}")
        continue

    print(f"  Rows: {len(df)}")
    print(f"  Date range: {df['timestamp'].min()} -> {df['timestamp'].max()}")
    display(df.head())

BinanceAPIException: APIError(code=-1003): Too much request weight used; current limit is 6000 request weight per 1 MINUTE. Please use WebSocket Streams for live updates to avoid polling the API.

In [5]:
# Compare with an existing file to make sure format matches
existing = pd.read_parquet(OUTPUT_DIR / "binance_AAVEUSDT_1h_daily_20260221_130805.parquet")
print("Existing file columns:", list(existing.columns))
print("New data columns:     ", list(df.columns))
print()
print("Existing dtypes:")
print(existing.dtypes)
print()
print("New dtypes:")
print(df.dtypes)

FileNotFoundError: [Errno 2] No such file or directory: '../data/binance_v2/binance_AAVEUSDT_1h_daily_20260221_130805.parquet'

In [7]:
# Save to parquet
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for symbol in SYMBOLS:
    df = fetch_1h_daily_klines(client, symbol, START_DATE, END_DATE)
    if df.empty:
        print(f"Skipping {symbol} - no data")
        continue

    timestamp_str = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
    filename = f"binance_{symbol}_1h_daily_{timestamp_str}.parquet"
    filepath = OUTPUT_DIR / filename
    df.to_parquet(filepath, index=False)
    print(f"Saved: {filepath} ({len(df)} rows)")

Saved: ../data/binance_v2/binance_FTTUSDT_1h_daily_20260221_132337.parquet (448 rows)


# Fetch Missing 1-Minute Perpetual Data (SKY)

Fetch 1m klines for symbols present in the v2 signal table but missing from
the existing 1m dataset at `binance_1m_perps_20241201_20260213/`.

In [9]:
# --- Configuration ---
SYMBOLS_1M = ["SKYUSDT"]
START_DATE_1M = "2024-12-01"
END_DATE_1M = datetime.now(UTC).strftime("%Y-%m-%d")
OUTPUT_DIR_1M = Path("../data/binance_1m_perps_20241201_20260213")

print(f"Symbols: {SYMBOLS_1M}")
print(f"Date range: {START_DATE_1M} to {END_DATE_1M}")
print(f"Output: {OUTPUT_DIR_1M.resolve()}")

Symbols: ['SKYUSDT']
Date range: 2024-12-01 to 2026-02-24
Output: /home/ra_yeye/2026_projects/nautilus_trader/my_strategies/data/binance_1m_perps_20241201_20260213


In [10]:
import time
from datetime import timedelta


def fetch_perp_1m_klines(client: Client, symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch 1m perpetual futures klines in 7-day chunks with retry to avoid rate limits."""
    all_dfs = []
    chunk_start = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    chunk_days = 7

    while chunk_start < end_dt:
        chunk_end = min(chunk_start + timedelta(days=chunk_days), end_dt)
        chunk_start_str = chunk_start.strftime("%Y-%m-%d")
        chunk_end_str = chunk_end.strftime("%Y-%m-%d")

        print(f"    Chunk: {chunk_start_str} -> {chunk_end_str}", end="", flush=True)

        # Retry with exponential backoff on rate limit
        for attempt in range(5):
            try:
                klines = client.get_historical_klines(
                    symbol=symbol,
                    interval=Client.KLINE_INTERVAL_1MINUTE,
                    start_str=f"{chunk_start_str} 00:00:00",
                    end_str=f"{chunk_end_str} 23:59:59",
                    klines_type=HistoricalKlinesType.FUTURES,
                )
                break
            except Exception as e:
                if "-1003" in str(e):
                    wait = 60 * (attempt + 1)
                    print(f" [rate limited, waiting {wait}s]", end="", flush=True)
                    time.sleep(wait)
                else:
                    raise
        else:
            print(" FAILED after 5 retries, skipping")
            chunk_start = chunk_end
            continue

        if klines:
            df = pd.DataFrame(klines, columns=[
                "timestamp", "open", "high", "low", "close", "volume",
                "close_time", "quote_volume", "trades_count", "taker_buy_volume",
                "taker_buy_quote_volume", "ignore",
            ])
            all_dfs.append(df)
            print(f" -> {len(df):,} rows", flush=True)
        else:
            print(" -> 0 rows", flush=True)

        chunk_start = chunk_end
        # Sleep 15s between chunks to stay under rate limit
        if chunk_start < end_dt:
            time.sleep(15)

    if not all_dfs:
        return pd.DataFrame()

    df = pd.concat(all_dfs, ignore_index=True)

    # Convert types
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)
    for col in ["open", "high", "low", "close", "volume", "quote_volume",
                "taker_buy_volume", "taker_buy_quote_volume"]:
        df[col] = df[col].astype(float)
    df["trades_count"] = df["trades_count"].astype(int)

    # Add metadata columns
    df["symbol"] = symbol
    df["interval"] = "1m"

    # Select columns in consistent order (matches api_binance_data.py)
    df = df[[
        "symbol", "interval", "timestamp", "close_time",
        "open", "high", "low", "close", "volume", "quote_volume",
        "taker_buy_volume", "taker_buy_quote_volume", "trades_count",
    ]].reset_index(drop=True)

    # Drop duplicates in case chunks overlap
    df = df.drop_duplicates(subset=["timestamp"], keep="first").reset_index(drop=True)

    return df

In [11]:
# Fetch and save 1m data
client_1m = Client()

for symbol in SYMBOLS_1M:
    print(f"Fetching {symbol} (1m)... this may take a while")
    df_1m = fetch_perp_1m_klines(client_1m, symbol, START_DATE_1M, END_DATE_1M)

    if df_1m.empty:
        print(f"  No data found for {symbol}")
        continue

    print(f"  Rows: {len(df_1m):,}")
    print(f"  Date range: {df_1m['timestamp'].min()} -> {df_1m['timestamp'].max()}")

    # Save as {SYMBOL}.parquet to match existing files in the directory
    filepath = OUTPUT_DIR_1M / f"{symbol}.parquet"
    df_1m.to_parquet(filepath, index=False)
    print(f"  Saved: {filepath}")

    display(df_1m.head())

BinanceAPIException: APIError(code=-1003): Too much request weight used; current limit is 6000 request weight per 1 MINUTE. Please use WebSocket Streams for live updates to avoid polling the API.